# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.05 · Cierre humano y snapshot entrenable

Reincorpora el último evento humano por chunk, recupera `video_id` desde el chunk fuente y congela un snapshot inmutable de cinco salidas.

La separación agrupada por video evita que fragmentos correlacionados del mismo video crucen particiones, una decisión de diseño coherente con el control de sesgo de selección [1]. Los snapshots, insumos y manifiestos usan SHA-256 [2]. La precedencia humana sigue siendo una regla local y no convierte automáticamente toda intervención en verdad objetiva [3].

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


## Reconciliación append-only

In [2]:
from tqdm.auto import tqdm
from moderacion_peru.consolidation import reconcile_human_reviews
CONSOLIDATED=ROOT/'datos/etiquetado/consolidado/anotaciones_v2.jsonl'
CHUNKS=ROOT/'datos/processed/chunks_v2.jsonl'
REVIEWS=[ROOT/'datos/etiquetado/humano/labeling_events_v2.jsonl']
REVIEWED=ROOT/'datos/etiquetado/consolidado/anotaciones_revisadas_v2.jsonl'
stage_progress={'bar':None}
STAGE_PHASES={'loading_chunks':'Cargando chunks','loading_review_events':'Leyendo eventos humanos','reconciling':'Reconciliando decisiones','loading_previous_snapshot':'Leyendo snapshot anterior','preparing_snapshot':'Preparando snapshot','deduplicating_snapshot':'Deduplicando snapshot','validating_video_splits':'Validando splits por video'}
def report_stage_progress(event):
    if event['status']=='phase_started':
        if stage_progress.get('bar') is not None:
            stage_progress['bar'].close()
        stage_progress['bar']=tqdm(total=event.get('total'),desc=STAGE_PHASES.get(event['phase'],event['phase']),unit='registro')
        return
    bar=stage_progress.get('bar')
    if bar is not None and event.get('advance'):
        bar.update(event['advance'])
        details={}
        if 'reviewed' in event:
            details['revisados']=event['reviewed']
        if 'eligible' in event:
            details['elegibles']=event['eligible']
        if details:
            bar.set_postfix(**details)
    if event['status']=='finished' and bar is not None:
        bar.close()
        stage_progress['bar']=None
try:
    reconciliation_result=reconcile_human_reviews(CONSOLIDATED,REVIEWS,REVIEWED,chunks_source=CHUNKS,progress_callback=report_stage_progress)
finally:
    if stage_progress.get('bar') is not None:
        stage_progress['bar'].close()
show_result('Reconciliación humana',reconciliation_result,tone='success')

human:reject,9221
without_human_event,126537
human:modify,44145
human:accept,2558
rows,182461
duplicate_events,0
orphan_events,0
status,noop
input_signature,52d54799f84016ad37e0a101a18e2d1ec514caf0d9c13d7ee1f5decbb9aea068


## Snapshot versionado

In [3]:
from moderacion_peru.datasets import materialize_versioned_training_snapshot
DATASET=ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
try:
    snapshot=materialize_versioned_training_snapshot(REVIEWED,DATASET,progress_callback=report_stage_progress)
finally:
    if stage_progress.get('bar') is not None:
        stage_progress['bar'].close()
show_result('Snapshot entrenable', snapshot, tone='success')
show_callout('Idempotencia', 'Sin cambios de entrada, ambas operaciones devuelven status=noop y no reescriben archivos.', tone='neutral')

Leyendo snapshot anterior: 0registro [00:00, ?registro/s]

Preparando snapshot: 0registro [00:00, ?registro/s]

Deduplicando snapshot:   0%|          | 0/173240 [00:00<?, ?registro/s]

Validando splits por video:   0%|          | 0/173240 [00:00<?, ?registro/s]

status,noop
snapshot_id,v2.1.0-e354b3248f7418f1
snapshot,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\model_ready\v2\snapshots\v2.1.0-e354b3248f7418f1\dataset_5_salidas.jsonl
manifest,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\model_ready\v2\snapshots\v2.1.0-e354b3248f7418f1\snapshot_manifest.json
canonical,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\model_ready\v2\dataset_5_salidas.jsonl
dataset_sha256,24d3d81d23c00cb1ba27fcca8c2759a97be5c9932ceaeb60be0a4e9a1cfca783
rows,173240
videos,4906


## Referencias

[1] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.

[2] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.

[3] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.